
# ARC-v0.15.1 — Full-Coverage Signed Harmful-Amplification Repair

This notebook closes the remaining directionality gap in ARC-v0.15.

ARC-v0.15 established that a deployable selector using only PQ32-side statistics and policy variables remains predictive on untouched FEVER validation. Its signed audit, however, used an older FEVER paired-trajectory artifact that covered only a small overlap with the full ARC-v0.13 validation grid.

ARC-v0.15.1 therefore reconstructs signed utility **only for the ARC-v0.13 validation query-policy rows already classified as absolute amplification**.

## Primary estimand

For every ARC-v0.13 validation row with

\[
H3_{abs} > 0.002,
\]

reconstruct

\[
G_t = nDCG_{SQ8}(t)-nDCG_{PQ32}(t)
\]

over rounds \(t=0,\ldots,4\), then estimate

\[
H3_{signed}=slope(G_t).
\]

The main paper-facing quantity is

\[
P(G_T>0 \mid H3_{abs}>0.002).
\]

## Design constraints

- source ARC-v0.13 FIT/validation artifacts remain read-only;
- no FEVER test access;
- no change to the frozen \(\epsilon=0.002\) regime threshold;
- no full 44-config validation rerun;
- replay only the ~8% validation query-policy trajectories already identified as amplifying;
- use the **exact v0.13 feedback helper functions extracted from the original v0.13 notebook**, rather than re-implementing the update rule from memory;
- every reconstructed config is checkpointed independently for resume safety;
- absolute-gap trajectories are checked against the sealed v0.13 checkpoint values before signed conclusions are accepted.


In [ ]:

# ============================================================
# Cell 1 — Environment, Drive, frozen source
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import ast
import gc
import hashlib
import inspect
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import faiss
except ImportError:
    !pip -q install faiss-cpu==1.12.0 pyarrow pandas
    import faiss

SEED = 20260816
DIM = 384
N_DOCS = 5_416_568
NPROBE = 64
TOP_RETRIEVE = 100
MAX_ROUNDS = 4
EPS = 0.002
TIE_EPS = 1e-12
FAISS_THREADS = 1

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed"

ROOT = (
    DRIVE_ROOT
    / "hc-rars-fever-5m-untouched-confirmation-v1"
)

ARC_ROOT = (
    DRIVE_ROOT
    / "rag-pq-checkpoints"
    / "arc-v0"
)

INDEX_ROOT = (
    DRIVE_ROOT
    / "rag-pq-checkpoints"
    / "arc-index-cache"
)

V013_RUN = (
    ARC_ROOT
    / "fever-boundary-external-replication-v013"
    / "20260817-140640"
)

assert V013_RUN.is_dir(), V013_RUN

PROTOCOL_PATH = V013_RUN / "v013_fever_boundary_protocol.json"
VALIDATION_REPORT_PATH = V013_RUN / "v013_validation_continuation_report.json"

fit_files = sorted(V013_RUN.glob("fit-*.parquet"))
val_files = sorted(V013_RUN.glob("validation-*.parquet"))

assert len(fit_files) == 44
assert len(val_files) == 44
assert PROTOCOL_PATH.is_file()
assert VALIDATION_REPORT_PATH.is_file()

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
validation_report = json.loads(
    VALIDATION_REPORT_PATH.read_text(encoding="utf-8")
)

assert protocol["test_access_allowed"] is False
assert validation_report["test_accessed"] is False
assert np.isclose(
    float(protocol["regime_threshold_abs_slope"]),
    EPS,
)

V0151_ROOT = (
    ARC_ROOT
    / "signed-harm-full-coverage-repair-v0151"
)
V0151_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V0151_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

CHECKPOINT_DIR = OUT / "signed-checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

print("Drive:", DRIVE_ROOT)
print("Frozen v0.13:", V013_RUN)
print("Validation checkpoints:", len(val_files))
print("Frozen EPS:", EPS)
print("Output:", OUT)
print("ARC-v0.15.1 PREFLIGHT — PASS")


In [ ]:

# ============================================================
# Cell 2 — Resolve original v0.13 notebook and extract exact
# feedback helper functions
#
# Robust version:
# - parses each code cell independently
# - skips IPython magics / shell commands
# - extracts exact original helpers:
#   cfg_key, fetch_doc_vectors, feedback_matrix, anchored_update
# ============================================================

NOTEBOOK_CANDIDATES = [
    DRIVE_ROOT / "Colab Notebooks" / "ARC_v0_13_Colab_FEVER_Boundary_External_Replication.ipynb",
    DRIVE_ROOT / "Colab Notebooks" / "ARC_v0_13_Colab_FEVER_Boundary_External_Replication(1).ipynb",
]

for p in DRIVE_ROOT.rglob(
    "ARC_v0_13_Colab_FEVER_Boundary_External_Replication*.ipynb"
):
    if p not in NOTEBOOK_CANDIDATES:
        NOTEBOOK_CANDIDATES.append(p)

V013_NOTEBOOK = next(
    (p for p in NOTEBOOK_CANDIDATES if p.is_file()),
    None,
)

assert V013_NOTEBOOK is not None, (
    "Original v0.13 notebook not found in Drive. "
    "Keep it in MyDrive/Colab Notebooks or update V013_NOTEBOOK."
)

nb013 = json.loads(
    V013_NOTEBOOK.read_text(
        encoding="utf-8"
    )
)

wanted_functions = {
    "cfg_key",
    "fetch_doc_vectors",
    "feedback_matrix",
    "anchored_update",
}

found = {}
parse_failures = []

for cell_index, cell in enumerate(
    nb013.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if not source.strip():
        continue

    cleaned_lines = []

    for line in source.splitlines():
        stripped = line.lstrip()

        # Notebook magics / shell commands are valid in IPython
        # but not valid Python syntax for ast.parse().
        if (
            stripped.startswith("%")
            or stripped.startswith("!")
            or stripped.startswith("?")
        ):
            continue

        cleaned_lines.append(line)

    cleaned_source = "\n".join(cleaned_lines)

    if not cleaned_source.strip():
        continue

    try:
        tree = ast.parse(cleaned_source)

    except SyntaxError as e:
        parse_failures.append({
            "cell_index": cell_index,
            "error": repr(e),
            "preview": cleaned_source[:300],
        })
        continue

    for node in tree.body:
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            if node.name in wanted_functions:
                found[node.name] = node

missing = wanted_functions - set(found)

if missing:
    print(
        "AST parse failures skipped:",
        len(parse_failures),
    )

    for item in parse_failures[:10]:
        print()
        print("CELL", item["cell_index"])
        print(item["error"])
        print(item["preview"])

assert not missing, (
    "Could not extract exact helper(s) from v0.13 notebook: "
    + ", ".join(sorted(missing))
)

# Compile in dependency-safe order.
helper_order = [
    "cfg_key",
    "fetch_doc_vectors",
    "feedback_matrix",
    "anchored_update",
]

module = ast.Module(
    body=[
        found[name]
        for name in helper_order
    ],
    type_ignores=[],
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        filename=str(V013_NOTEBOOK),
        mode="exec",
    ),
    globals(),
)

print("v0.13 notebook:", V013_NOTEBOOK)
print(
    "AST parse failures skipped:",
    len(parse_failures),
)

for name in helper_order:
    fn = globals()[name]

    print()
    print(
        f"{name:20s}",
        inspect.signature(fn),
    )

    print(
        "global names:",
        sorted(
            set(
                fn.__code__.co_names
            )
        ),
    )

print()
print(
    "EXACT V0.13 HELPER EXTRACTION — PASS"
)


In [ ]:

# ============================================================
# Cell 3 — Load exact FEVER DEV state required by v0.13
# ============================================================

CORPUS_MEMMAP = (
    ROOT / "stage1/corpus_embeddings.float16.memmap"
)

QUERY_EMB = (
    ROOT / "stage1/query_embeddings_v2.float32.npy"
)

QUERY_IDS = (
    ROOT / "stage1/query_ids.utf8.txt"
)

SPLIT_MANIFEST = (
    ROOT / "stage1/official_split_manifest.json"
)

DEV_QRELS = (
    ROOT / "stage2/dev_qrels_rows.csv"
)

PQ32_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
)

SQ8_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss"
)

required = {
    "CORPUS_MEMMAP": CORPUS_MEMMAP,
    "QUERY_EMB": QUERY_EMB,
    "QUERY_IDS": QUERY_IDS,
    "SPLIT_MANIFEST": SPLIT_MANIFEST,
    "DEV_QRELS": DEV_QRELS,
    "PQ32_INDEX": PQ32_PATH,
    "SQ8_INDEX": SQ8_PATH,
}

for name, path in required.items():
    print(
        f"{name:20s}",
        "OK" if path.is_file() else "MISSING",
        path,
    )
    assert path.is_file(), path

queries = np.load(
    QUERY_EMB,
    mmap_mode="r",
)

with open(
    QUERY_IDS,
    "r",
    encoding="utf-8",
) as f:
    query_ids = [
        x.strip()
        for x in f
        if x.strip()
    ]

with open(
    SPLIT_MANIFEST,
    "r",
    encoding="utf-8",
) as f:
    split = json.load(f)

dev_ids = [
    str(x)
    for x in split["dev_query_ids"]
]

assert len(dev_ids) == 6666

for key in [
    "test_qrels_relevance_values_accessed",
    "test_retrieval_performed",
    "test_outcomes_observed",
]:
    if key in split:
        assert split[key] is False, (
            key,
            split[key],
        )

query_row = {
    str(qid): i
    for i, qid
    in enumerate(query_ids)
}

missing_queries = [
    q
    for q in dev_ids
    if q not in query_row
]

assert not missing_queries, (
    missing_queries[:10]
)

dev_rows = np.asarray(
    [
        query_row[q]
        for q in dev_ids
    ],
    dtype=np.int64,
)

Q_DEV = np.asarray(
    queries[dev_rows],
    dtype=np.float32,
)

assert Q_DEV.shape == (
    6666,
    DIM,
)

# Match the original v0.13 normalization.
Q_DEV /= np.maximum(
    np.linalg.norm(
        Q_DEV,
        axis=1,
        keepdims=True,
    ),
    1e-12,
)

# FEVER corpus embedding memmap.
corpus = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(
        N_DOCS,
        DIM,
    ),
)

# Aliases deliberately exposed because exact v0.13 helpers are
# dynamically extracted and may reference one of these names.
CORPUS = corpus
corpus_embeddings = corpus
corpus_memmap = corpus


# ============================================================
# Robust FEVER qrels loader
#
# query-id   : FEVER query identifier
# corpus-id  : textual document identifier/title
# corpus-row : integer row returned by FAISS
# score      : relevance
#
# CRITICAL: use corpus-row, NOT corpus-id.
# ============================================================

qrels_raw = pd.read_csv(
    DEV_QRELS
)

print(
    "Qrels columns:",
    list(qrels_raw.columns),
)

qid_candidates = [
    "query_id",
    "qid",
    "query-id",
    "query",
]

qid_col = next(
    (
        c
        for c in qid_candidates
        if c in qrels_raw.columns
    ),
    None,
)

assert qid_col is not None, (
    "Cannot identify query-id column.",
    list(qrels_raw.columns),
)

row_candidates = [
    "corpus-row",
    "corpus_row",
    "row_id",
    "row",
]

row_col = next(
    (
        c
        for c in row_candidates
        if c in qrels_raw.columns
    ),
    None,
)

assert row_col is not None, (
    "Cannot identify FAISS corpus-row column.",
    list(qrels_raw.columns),
)

rel_candidates = [
    "score",
    "relevance",
    "rel",
    "label",
]

rel_col = next(
    (
        c
        for c in rel_candidates
        if c in qrels_raw.columns
    ),
    None,
)

if rel_col is None:
    qrels_raw["_rel"] = 1.0
    rel_col = "_rel"

qrels_raw[qid_col] = (
    qrels_raw[qid_col]
    .astype(str)
)

qrels_raw[row_col] = (
    pd.to_numeric(
        qrels_raw[row_col],
        errors="raise",
    )
    .astype(np.int64)
)

qrels_raw[rel_col] = (
    pd.to_numeric(
        qrels_raw[rel_col],
        errors="raise",
    )
    .astype(float)
)

print()
print("qid column :", qid_col)
print("row column :", row_col)
print("rel column :", rel_col)

dev_qrels = {}

for qid, g in qrels_raw.groupby(
    qid_col,
    sort=False,
):
    rel_docs = set(
        int(row)
        for row, rel
        in zip(
            g[row_col],
            g[rel_col],
        )
        if float(rel) > 0
    )

    dev_qrels[
        str(qid)
    ] = rel_docs

# Common aliases.
DEV_QRELS_DICT = dev_qrels
qrels = dev_qrels

all_rel_rows = [
    row
    for docs in dev_qrels.values()
    for row in docs
]

assert all_rel_rows
assert min(all_rel_rows) >= 0
assert max(all_rel_rows) < N_DOCS

print()
print(
    "DEV qrels queries:",
    len(dev_qrels),
)
print(
    "Relevant corpus-row min:",
    min(all_rel_rows),
)
print(
    "Relevant corpus-row max:",
    max(all_rel_rows),
)

print()
print("Q_DEV:", Q_DEV.shape)
print("Corpus:", corpus.shape, corpus.dtype)
print(
    "FEVER QRELS ROW-ID MAPPING — PASS"
)
print(
    "FEVER STATE LOAD — PASS"
)


In [ ]:

# ============================================================
# Cell 4 — Load frozen v0.13 validation and identify every
# absolute-amplification query-config row
# ============================================================

val_frames = []

for i, p in enumerate(val_files, 1):
    df = pd.read_parquet(p)
    val_frames.append(df)

val_traj = pd.concat(
    val_frames,
    ignore_index=True,
)

del val_frames
gc.collect()

GROUP_COLS = [
    "query_id",
    "low",
    "high",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

def frozen_abs_slopes(df):
    rows = []

    for keys, g in df.groupby(
        GROUP_COLS,
        dropna=False,
        sort=False,
    ):
        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)
        y = g["abs_utility_gap"].to_numpy(np.float64)

        row = dict(zip(GROUP_COLS, keys))
        row["H3_abs_slope"] = float(
            np.polyfit(x, y, 1)[0]
        )

        rows.append(row)

    return pd.DataFrame(rows)

val_slopes = frozen_abs_slopes(val_traj)

assert len(val_slopes) == 3316 * 44

amp_rows = (
    val_slopes.loc[
        val_slopes["H3_abs_slope"] > EPS
    ]
    .copy()
    .reset_index(drop=True)
)

print("Validation slope rows:", len(val_slopes))
print("Absolute-amplifying rows:", len(amp_rows))
print(
    "Amplifying prevalence:",
    len(amp_rows) / len(val_slopes),
)

assert len(amp_rows) == 11596, (
    "Expected the sealed v0.13 validation amplification count 11,596; "
    f"got {len(amp_rows)}."
)

amp_rows.to_parquet(
    OUT / "v0151_target_absolute_amplification_rows.parquet",
    index=False,
)

print("FULL AMPLIFICATION TARGET SET — PASS")


In [ ]:

# ============================================================
# Cell 5 — Exact utility metric + helper dependency gate
#
# Robust dependency validation:
# - Python builtins such as ValueError are allowed
# - NumPy attribute names such as norm are not treated as
#   standalone missing globals
# - fetch_doc_vectors is extracted directly from v0.13
# ============================================================

import builtins

DISCOUNTS = (
    1.0
    / np.log2(
        np.arange(
            2,
            12,
        )
    )
)

def ndcg_at_10_exact(
    qids,
    ids,
):
    out = np.zeros(
        len(qids),
        np.float32,
    )

    for i, qid in enumerate(
        qids
    ):
        rel = dev_qrels.get(
            str(qid),
            set(),
        )

        hits = np.asarray(
            [
                int(doc) in rel
                for doc in ids[i, :10]
            ],
            dtype=np.float64,
        )

        dcg = float(
            (
                hits
                * DISCOUNTS
            ).sum()
        )

        ideal = min(
            len(rel),
            10,
        )

        idcg = float(
            DISCOUNTS[:ideal].sum()
        )

        out[i] = (
            dcg / idcg
            if idcg > 0
            else 0.0
        )

    return out


# ============================================================
# Helper availability
# ============================================================

required_helpers = [
    "cfg_key",
    "fetch_doc_vectors",
    "feedback_matrix",
    "anchored_update",
]

for name in required_helpers:
    assert name in globals(), (
        f"Missing extracted v0.13 helper: {name}"
    )

    assert callable(
        globals()[name]
    ), (
        f"{name} exists but is not callable"
    )

print(
    "Extracted helper availability — PASS"
)


# ============================================================
# Dependency checker
# ============================================================

def check_function_globals(fn):
    names = set(
        fn.__code__.co_names
    )

    missing = []

    for name in names:
        if name in globals():
            continue

        if hasattr(
            builtins,
            name,
        ):
            continue

        if hasattr(
            np,
            name,
        ):
            continue

        if hasattr(
            np.linalg,
            name,
        ):
            continue

        # Common ndarray / object methods that appear in co_names.
        if name in {
            "reshape",
            "astype",
            "sum",
            "max",
            "copy",
            "clip",
            "exp",
            "maximum",
            "asarray",
            "mean",
            "sqrt",
        }:
            continue

        missing.append(name)

    return sorted(missing)


for fn_name in [
    "fetch_doc_vectors",
    "feedback_matrix",
    "anchored_update",
]:
    fn = globals()[fn_name]

    missing = check_function_globals(
        fn
    )

    print(
        f"{fn_name:20s}",
        "unresolved globals:",
        missing,
    )

    assert not missing, (
        fn_name,
        missing,
        (
            "A real source-level dependency is still missing. "
            "Inspect the original v0.13 helper before replay."
        ),
    )


# ============================================================
# fetch_doc_vectors smoke test
# ============================================================

test_ids = np.asarray(
    [
        [0, 1, 2],
        [3, 4, 5],
    ],
    dtype=np.int64,
)

test_vec = fetch_doc_vectors(
    test_ids
)

print()
print(
    "fetch_doc_vectors output shape:",
    test_vec.shape,
)

print(
    "fetch_doc_vectors dtype:",
    test_vec.dtype,
)

assert test_vec.shape == (
    2,
    3,
    DIM,
), test_vec.shape

assert np.isfinite(
    test_vec
).all()

faiss.omp_set_num_threads(
    FAISS_THREADS
)

print()
print(
    "UTILITY + HELPER DEPENDENCY GATE — PASS"
)


In [ ]:

# ============================================================
# Cell 6 — Replay one amplifying config with signed utilities
#
# This cell both reconstructs G_t and checks that the newly
# computed |G_t| matches the sealed v0.13 abs_utility_gap.
# ============================================================

dev_pos = {
    str(qid): i
    for i, qid in enumerate(dev_ids)
}

def config_from_row(row):
    temp = row["temperature"]

    if pd.isna(temp):
        temp = None
    else:
        temp = float(temp)

    return {
        "method": str(row["method"]),
        "alpha": float(row["alpha"]),
        "k": int(row["k"]),
        "temperature": temp,
    }

def replay_signed_for_config(config_row, target_qids):
    cfg = config_from_row(config_row)

    idx = np.asarray(
        [dev_pos[str(q)] for q in target_qids],
        dtype=np.int64,
    )

    qids = [str(dev_ids[i]) for i in idx]
    q0 = np.asarray(
        Q_DEV[idx],
        dtype=np.float32,
    ).copy()

    low = faiss.read_index(str(PQ32_PATH))
    high = faiss.read_index(str(SQ8_PATH))

    low.nprobe = NPROBE
    high.nprobe = NPROBE

    qL = q0.copy()
    qH = q0.copy()

    frames = []

    for t in range(MAX_ROUNDS + 1):
        sL, idL = low.search(
            np.ascontiguousarray(qL, np.float32),
            TOP_RETRIEVE,
        )

        sH, idH = high.search(
            np.ascontiguousarray(qH, np.float32),
            TOP_RETRIEVE,
        )

        nL = ndcg_at_10_exact(qids, idL)
        nH = ndcg_at_10_exact(qids, idH)

        frames.append(
            pd.DataFrame({
                "query_id": qids,
                "iteration": t,
                "pq32_ndcg": nL,
                "sq8_ndcg": nH,
                "G_signed": nH - nL,
                "abs_utility_gap_reconstructed":
                    np.abs(nH - nL),
                "config_key": str(config_row["config_key"]),
            })
        )

        if t < MAX_ROUNDS:
            fL = feedback_matrix(
                idL,
                sL,
                cfg,
            )

            fH = feedback_matrix(
                idH,
                sH,
                cfg,
            )

            qL = anchored_update(
                q0,
                fL,
                cfg["alpha"],
            )

            qH = anchored_update(
                q0,
                fH,
                cfg["alpha"],
            )

    del low, high
    gc.collect()

    return pd.concat(
        frames,
        ignore_index=True,
    )

# Smoke test using the first config with at least one amplification.
smoke_cfg_key = amp_rows.iloc[0]["config_key"]

smoke_meta = (
    amp_rows.loc[
        amp_rows["config_key"] == smoke_cfg_key
    ]
    .iloc[0]
)

smoke_qids = (
    amp_rows.loc[
        amp_rows["config_key"] == smoke_cfg_key,
        "query_id",
    ]
    .astype(str)
    .tolist()[: min(32, int(
        (
            amp_rows["config_key"]
            == smoke_cfg_key
        ).sum()
    ))]
)

smoke = replay_signed_for_config(
    smoke_meta,
    smoke_qids,
)

sealed_smoke = (
    val_traj.loc[
        (
            val_traj["config_key"]
            == smoke_cfg_key
        )
        &
        (
            val_traj["query_id"]
            .astype(str)
            .isin(smoke_qids)
        ),
        [
            "query_id",
            "iteration",
            "config_key",
            "abs_utility_gap",
        ],
    ]
    .copy()
)

sealed_smoke["query_id"] = (
    sealed_smoke["query_id"]
    .astype(str)
)

smoke_check = smoke.merge(
    sealed_smoke,
    on=[
        "query_id",
        "iteration",
        "config_key",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(smoke_check) == len(smoke)

smoke_diff = np.abs(
    smoke_check[
        "abs_utility_gap_reconstructed"
    ].to_numpy(np.float64)
    -
    smoke_check[
        "abs_utility_gap"
    ].to_numpy(np.float64)
)

print("Smoke config:", smoke_cfg_key)
print("Smoke queries:", len(smoke_qids))
print("max |reconstructed abs-gap - sealed|:", float(smoke_diff.max()))
print("mean difference:", float(smoke_diff.mean()))

# nDCG is stored/recomputed in float32 in the original FEVER pipeline.
ABS_GAP_TOL = 2e-6

assert float(smoke_diff.max()) <= ABS_GAP_TOL, (
    "Replay does not reproduce the sealed v0.13 absolute utility gap.",
    float(smoke_diff.max()),
    ABS_GAP_TOL,
)

print("SIGNED REPLAY SMOKE TEST — PASS")


In [ ]:

# ============================================================
# Cell 7 — Resume-safe signed replay for ALL 11,596
# absolute-amplifying validation query-config rows
# ============================================================

config_meta = (
    amp_rows[
        [
            "config_key",
            "method",
            "alpha",
            "k",
            "temperature",
        ]
    ]
    .drop_duplicates("config_key")
    .sort_values("config_key")
    .reset_index(drop=True)
)

print("Configs containing amplification:", len(config_meta))
print("Target amplifying rows:", len(amp_rows))

manifest_rows = []

for i, row in config_meta.iterrows():
    cfgk = str(row["config_key"])

    target_qids = (
        amp_rows.loc[
            amp_rows["config_key"] == cfgk,
            "query_id",
        ]
        .astype(str)
        .tolist()
    )

    checkpoint = (
        CHECKPOINT_DIR
        / f"signed-{cfgk}.parquet"
    )

    print()
    print("=" * 80)
    print(
        f"[SIGNED {i+1:02d}/{len(config_meta):02d}]",
        cfgk,
        "queries=",
        len(target_qids),
    )

    if checkpoint.is_file():
        df = pd.read_parquet(checkpoint)

        expected_rows = len(target_qids) * (MAX_ROUNDS + 1)

        if (
            len(df) == expected_rows
            and df["query_id"].astype(str).nunique()
                == len(target_qids)
            and df["iteration"].nunique()
                == MAX_ROUNDS + 1
        ):
            print("RESUME — existing checkpoint valid")
        else:
            print("Existing checkpoint invalid; rebuilding")
            checkpoint.unlink()
            df = None
    else:
        df = None

    if df is None:
        t0 = time.perf_counter()

        df = replay_signed_for_config(
            row,
            target_qids,
        )

        dt = time.perf_counter() - t0

        # Exact replay validation against sealed v0.13 abs gap.
        sealed = (
            val_traj.loc[
                (
                    val_traj["config_key"]
                    == cfgk
                )
                &
                (
                    val_traj["query_id"]
                    .astype(str)
                    .isin(target_qids)
                ),
                [
                    "query_id",
                    "iteration",
                    "config_key",
                    "abs_utility_gap",
                ],
            ]
            .copy()
        )

        sealed["query_id"] = (
            sealed["query_id"]
            .astype(str)
        )

        chk = df.merge(
            sealed,
            on=[
                "query_id",
                "iteration",
                "config_key",
            ],
            how="inner",
            validate="one_to_one",
        )

        assert len(chk) == len(df)

        diffs = np.abs(
            chk["abs_utility_gap_reconstructed"]
            .to_numpy(np.float64)
            -
            chk["abs_utility_gap"]
            .to_numpy(np.float64)
        )

        max_diff = float(diffs.max())
        mean_diff = float(diffs.mean())

        print("seconds:", dt)
        print("max abs-gap replay diff:", max_diff)
        print("mean abs-gap replay diff:", mean_diff)

        assert max_diff <= ABS_GAP_TOL, (
            cfgk,
            max_diff,
            ABS_GAP_TOL,
        )

        df.to_parquet(
            checkpoint,
            index=False,
        )

    manifest_rows.append({
        "config_key": cfgk,
        "target_queries": len(target_qids),
        "trajectory_rows": len(df),
        "checkpoint": str(checkpoint),
        "sha256": sha256_file(checkpoint),
    })

    print("SAVED/VALID:", checkpoint.name)

manifest = pd.DataFrame(manifest_rows)

assert int(manifest["target_queries"].sum()) == 11596

manifest.to_csv(
    OUT / "v0151_signed_checkpoint_manifest.csv",
    index=False,
)

print()
print("FULL-COVERAGE SIGNED REPLAY — PASS")


In [ ]:

# ============================================================
# Cell 8 — Combine signed checkpoints and reconstruct signed H3
# ============================================================

signed_frames = []

for p in sorted(CHECKPOINT_DIR.glob("signed-*.parquet")):
    signed_frames.append(
        pd.read_parquet(p)
    )

signed_traj = pd.concat(
    signed_frames,
    ignore_index=True,
)

del signed_frames
gc.collect()

expected_signed_rows = (
    len(amp_rows)
    * (MAX_ROUNDS + 1)
)

print("Signed trajectory rows:", len(signed_traj))
print("Expected:", expected_signed_rows)
print(
    "Unique query-config pairs:",
    signed_traj[
        ["query_id", "config_key"]
    ]
    .drop_duplicates()
    .shape[0],
)

assert len(signed_traj) == expected_signed_rows

assert (
    signed_traj[
        ["query_id", "config_key"]
    ]
    .drop_duplicates()
    .shape[0]
    == 11596
)

signed_rows = []

for keys, g in signed_traj.groupby(
    ["query_id", "config_key"],
    sort=False,
):
    qid, cfgk = keys

    g = g.sort_values("iteration")

    x = g["iteration"].to_numpy(np.float64)
    y = g["G_signed"].to_numpy(np.float64)

    signed_rows.append({
        "query_id": str(qid),
        "config_key": str(cfgk),
        "G0": float(y[0]),
        "GT": float(y[-1]),
        "delta_G": float(y[-1] - y[0]),
        "H3_signed_slope":
            float(np.polyfit(x, y, 1)[0]),
    })

signed_summary_rows = pd.DataFrame(
    signed_rows
)

amp_for_merge = amp_rows.copy()
amp_for_merge["query_id"] = (
    amp_for_merge["query_id"]
    .astype(str)
)

combined = amp_for_merge.merge(
    signed_summary_rows,
    on=[
        "query_id",
        "config_key",
    ],
    how="left",
    validate="one_to_one",
)

assert len(combined) == 11596

assert combined[
    [
        "G0",
        "GT",
        "delta_G",
        "H3_signed_slope",
    ]
].notna().all().all()

print("SIGNED H3 FULL-COVERAGE MERGE — PASS")


In [ ]:

# ============================================================
# Cell 9 — Primary signed taxonomy
# ============================================================

combined["signed_taxonomy"] = "unresolved_or_tied"

combined.loc[
    combined["GT"] > TIE_EPS,
    "signed_taxonomy",
] = "harmful_amplification"

combined.loc[
    combined["GT"] < -TIE_EPS,
    "signed_taxonomy",
] = "beneficial_divergence"

taxonomy = (
    combined[
        "signed_taxonomy"
    ]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="count")
)

taxonomy["fraction"] = (
    taxonomy["count"]
    / taxonomy["count"].sum()
)

p_gt_positive = float(
    (
        combined["GT"]
        > TIE_EPS
    ).mean()
)

p_signed_slope_positive = float(
    (
        combined["H3_signed_slope"]
        > 0
    ).mean()
)

p_delta_positive = float(
    (
        combined["delta_G"]
        > 0
    ).mean()
)

print("FULL-COVERAGE TAXONOMY")
display(taxonomy)

print(
    "P(G_T > 0 | H3_abs > EPS):",
    p_gt_positive,
)

print(
    "P(H3_signed > 0 | H3_abs > EPS):",
    p_signed_slope_positive,
)

print(
    "P(delta_G > 0 | H3_abs > EPS):",
    p_delta_positive,
)

combined.to_parquet(
    OUT
    / "v0151_full_coverage_signed_amplification_rows.parquet",
    index=False,
)

taxonomy.to_csv(
    OUT
    / "v0151_full_coverage_signed_taxonomy.csv",
    index=False,
)

print("PRIMARY SIGNED TAXONOMY — COMPLETE")


In [ ]:

# ============================================================
# Cell 10 — Query-cluster bootstrap CIs for signed proportions
# ============================================================

BOOTSTRAP_REPS = 2000

groups = {
    qid: g.copy()
    for qid, g in combined.groupby(
        "query_id",
        sort=False,
    )
}

qids = np.asarray(
    list(groups.keys()),
    dtype=object,
)

rng = np.random.default_rng(SEED)

boot_gt = []
boot_slope = []
boot_delta = []

for _ in range(BOOTSTRAP_REPS):
    sampled = rng.choice(
        qids,
        size=len(qids),
        replace=True,
    )

    frames = [
        groups[qid]
        for qid in sampled
    ]

    b = pd.concat(
        frames,
        ignore_index=True,
    )

    boot_gt.append(
        float(
            (
                b["GT"]
                > TIE_EPS
            ).mean()
        )
    )

    boot_slope.append(
        float(
            (
                b["H3_signed_slope"]
                > 0
            ).mean()
        )
    )

    boot_delta.append(
        float(
            (
                b["delta_G"]
                > 0
            ).mean()
        )
    )

bootstrap_summary = pd.DataFrame([
    {
        "metric":
            "P(GT>0 | H3_abs>EPS)",
        "estimate":
            p_gt_positive,
        "ci_low":
            float(np.quantile(boot_gt, 0.025)),
        "ci_high":
            float(np.quantile(boot_gt, 0.975)),
        "bootstrap_reps":
            BOOTSTRAP_REPS,
    },
    {
        "metric":
            "P(H3_signed>0 | H3_abs>EPS)",
        "estimate":
            p_signed_slope_positive,
        "ci_low":
            float(np.quantile(boot_slope, 0.025)),
        "ci_high":
            float(np.quantile(boot_slope, 0.975)),
        "bootstrap_reps":
            BOOTSTRAP_REPS,
    },
    {
        "metric":
            "P(delta_G>0 | H3_abs>EPS)",
        "estimate":
            p_delta_positive,
        "ci_low":
            float(np.quantile(boot_delta, 0.025)),
        "ci_high":
            float(np.quantile(boot_delta, 0.975)),
        "bootstrap_reps":
            BOOTSTRAP_REPS,
    },
])

display(bootstrap_summary)

bootstrap_summary.to_csv(
    OUT
    / "v0151_signed_query_cluster_bootstrap.csv",
    index=False,
)

print("SIGNED QUERY-CLUSTER BOOTSTRAP — PASS")


In [ ]:

# ============================================================
# Cell 11 — Policy-family / alpha directionality breakdown
# ============================================================

policy_summary = (
    combined
    .groupby(
        [
            "method",
            "alpha",
        ],
        as_index=False,
    )
    .agg(
        amplification_events=(
            "H3_abs_slope",
            "size",
        ),
        harmful_fraction=(
            "GT",
            lambda x: float(
                (x > TIE_EPS).mean()
            ),
        ),
        beneficial_fraction=(
            "GT",
            lambda x: float(
                (x < -TIE_EPS).mean()
            ),
        ),
        signed_slope_positive_fraction=(
            "H3_signed_slope",
            lambda x: float(
                (x > 0).mean()
            ),
        ),
        mean_GT=(
            "GT",
            "mean",
        ),
        mean_delta_G=(
            "delta_G",
            "mean",
        ),
    )
)

display(policy_summary)

policy_summary.to_csv(
    OUT
    / "v0151_signed_direction_by_policy_alpha.csv",
    index=False,
)

config_summary = (
    combined
    .groupby(
        "config_key",
        as_index=False,
    )
    .agg(
        amplification_events=(
            "H3_abs_slope",
            "size",
        ),
        harmful_fraction=(
            "GT",
            lambda x: float(
                (x > TIE_EPS).mean()
            ),
        ),
        beneficial_fraction=(
            "GT",
            lambda x: float(
                (x < -TIE_EPS).mean()
            ),
        ),
        mean_GT=("GT", "mean"),
        mean_delta_G=("delta_G", "mean"),
    )
)

config_summary.to_csv(
    OUT
    / "v0151_signed_direction_by_config.csv",
    index=False,
)

print("POLICY/CONFIG DIRECTIONALITY — COMPLETE")


In [ ]:

# ============================================================
# Cell 12 — Compare full-coverage result to ARC-v0.15 subset
# ============================================================

V015_ROOT = (
    ARC_ROOT
    / "deployable-boundary-signed-harm-audit-v015"
)

v015_runs = sorted(
    [
        p
        for p in V015_ROOT.iterdir()
        if p.is_dir()
    ],
    reverse=True,
) if V015_ROOT.is_dir() else []

subset_fraction = np.nan
subset_rows = np.nan

for run in v015_runs:
    p = (
        run
        / "v015_signed_harmful_amplification_summary.csv"
    )

    if p.is_file():
        old = pd.read_csv(p)

        if "harmful_amplification" in set(old["category"]):
            row = old.loc[
                old["category"]
                == "harmful_amplification"
            ].iloc[0]

            subset_fraction = float(
                row["fraction"]
            )

            subset_rows = int(
                old["count"].sum()
            )

            print("ARC-v0.15 subset source:", p)
            break

comparison = pd.DataFrame([{
    "v015_subset_events":
        subset_rows,
    "v015_subset_harmful_fraction":
        subset_fraction,
    "v0151_full_events":
        len(combined),
    "v0151_full_harmful_fraction":
        p_gt_positive,
    "absolute_fraction_difference":
        (
            abs(
                p_gt_positive
                - subset_fraction
            )
            if np.isfinite(subset_fraction)
            else np.nan
        ),
}])

display(comparison)

comparison.to_csv(
    OUT
    / "v0151_subset_vs_full_coverage_comparison.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 13 — Seal final report
# ============================================================

report = {
    "status":
        "ARC_V0151_FULL_COVERAGE_SIGNED_HARM_REPAIR_COMPLETE",

    "analysis_scope":
        (
            "Signed utility reconstruction for every ARC-v0.13 "
            "untouched-validation query-policy row with frozen "
            "H3_abs_slope > 0.002. Replay is restricted to those "
            "previously identified amplification rows."
        ),

    "source_v013_run":
        str(V013_RUN),

    "source_v013_protocol_sha256":
        sha256_file(PROTOCOL_PATH),

    "source_v013_validation_report_sha256":
        sha256_file(VALIDATION_REPORT_PATH),

    "source_v013_notebook":
        str(V013_NOTEBOOK),

    "source_v013_notebook_sha256":
        sha256_file(V013_NOTEBOOK),

    "exact_helpers_extracted": [
        "feedback_matrix",
        "anchored_update",
        "cfg_key",
    ],

    "regime_threshold_abs_slope":
        EPS,

    "full_validation_query_config_rows":
        int(len(val_slopes)),

    "target_absolute_amplification_rows":
        int(len(amp_rows)),

    "signed_rows_reconstructed":
        int(len(combined)),

    "signed_coverage_fraction":
        float(
            len(combined)
            / len(amp_rows)
        ),

    "P_GT_positive_given_abs_amplification":
        p_gt_positive,

    "P_H3_signed_positive_given_abs_amplification":
        p_signed_slope_positive,

    "P_delta_G_positive_given_abs_amplification":
        p_delta_positive,

    "retrieval_scope":
        (
            "Targeted replay of only pre-identified validation "
            "amplification trajectories; not a full validation sweep."
        ),

    "test_accessed":
        False,

    "interpretation_constraint":
        (
            "GT>0 means SQ8 has higher final nDCG@10 than PQ32 "
            "for the replayed synchronized feedback trajectories. "
            "This supports harmful lower-fidelity divergence but "
            "does not establish causal mediation."
        ),

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

assert report["signed_coverage_fraction"] == 1.0

REPORT_PATH = (
    OUT
    / "v0151_full_coverage_signed_harm_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(
    REPORT_PATH
)

(
    OUT
    / "V0151_REPORT_SHA256.txt"
).write_text(
    report_sha
    + "  "
    + REPORT_PATH.name
    + "\n",
    encoding="utf-8",
)

print()
print("=" * 80)
print("ARC-v0.15.1 FULL-COVERAGE SIGNED AUDIT — PASS")
print("=" * 80)
print("Output:", OUT)
print("Report SHA-256:", report_sha)
print("Coverage:", report["signed_coverage_fraction"])
print(
    "P(GT>0 | abs amplification):",
    p_gt_positive,
)
print(
    "P(H3_signed>0 | abs amplification):",
    p_signed_slope_positive,
)
print(
    "P(delta_G>0 | abs amplification):",
    p_delta_positive,
)
print("Test accessed:", False)
print("=" * 80)



## Paper-facing interpretation

Use the result conservatively.

If the full-coverage estimate remains high, e.g.

\[
P(G_T>0\mid H3_{abs}>0.002) \gg 0.5,
\]

the paper may state that **most absolute-gap amplification events correspond to trajectories in which the higher-fidelity SQ8 system ends with higher utility than PQ32**.

Prefer:

> “Most detected amplification events are directionally harmful to the lower-fidelity trajectory.”

Avoid:

> “Approximation always causes harmful error amplification.”

If the estimate is moderate or heterogeneous across policy families, use the broader term:

> **approximation-induced trajectory divergence**

and report the harmful subset explicitly.
